In [1]:
import sys
import os

current_file_path = os.path.abspath(os.getcwd()) # Funciona solo para JupyterNotebooks

# Obtener la ruta del directorio raíz del proyecto
project_root = os.path.abspath(os.path.join(os.path.dirname(current_file_path)))

# Añadir el directorio raíz del proyecto al sys.path
sys.path.append(project_root)

from config import INPUTS_FOLDER

statement_path = os.path.join(INPUTS_FOLDER, 'test_files', 'Nu', 'Nu_debit_statement.pdf')

### Pipeline description

In [2]:
from document_reader import PDFReader
from bank_detector import DefaultBankDetector
import logging
logging.getLogger('pdfminer').setLevel(logging.ERROR) # Abvoid PDFMiner warnings

bank_detector = DefaultBankDetector(PDFReader(statement_path))

extracted_words = bank_detector.get_extracted_words()
extracted_words

,page,text,x0,top,x1,bottom
0,1,Mario,399.800,44.910,426.110,54.910
1,1,Alexis,428.760,44.910,456.370,54.910
2,1,Canudas,459.020,44.910,500.110,54.910
3,1,Zertuche,502.760,44.910,545.000,54.910
4,1,Cuenta,427.310,59.910,461.160,69.910
...,...,...,...,...,...,...
2145,9,099,342.736,803.528,358.104,811.528
2146,9,1133,360.224,803.528,376.272,811.528
2147,9,9,521.280,815.528,526.240,823.528
2148,9,de,528.360,815.528,537.920,823.528


In [3]:
bank = bank_detector.detect_bank()
statement_type = bank_detector.detect_statement_type()
statement_properties = bank_detector.get_statement_properties()

print(f'{bank} - {statement_type}')

nu - debit


In [4]:
from table_boundary_detector import TransactionTableBoundaryDetector

boundary_detector = TransactionTableBoundaryDetector(bank_detector)

start_idx = boundary_detector.start_idx
end_idx = boundary_detector.end_idx

if statement_type == 'credit' and (start_idx is None or end_idx is None):
    print('New format')
    bank_detector = DefaultBankDetector(PDFReader(statement_path), new_credit_format=True)
    boundary_detector = TransactionTableBoundaryDetector(bank_detector)
    
    start_idx = boundary_detector.start_idx
    end_idx = boundary_detector.end_idx

filtered_words = boundary_detector.get_filtered_table_words()

print(f'{start_idx} - {end_idx}')
filtered_words  

149 - 1339


,page,text,x0,top,x1,bottom
0,2,FECHA,58.00,110.84,90.46,120.84
1,2,DEL,137.84,110.84,155.81,120.84
2,2,01,158.46,110.84,169.26,120.84
3,2,AL,171.91,110.84,184.00,120.84
4,2,30 SEP 2024,186.65,110.84,246.60,120.84
...,...,...,...,...,...,...
1185,5,Cajita,265.74,583.00,292.49,593.00
1186,5,"+$2,378.00",485.38,584.00,539.00,594.00
1187,5,Con,87.92,651.22,107.24,661.22
1188,5,estos,109.89,651.22,134.76,661.22


In [5]:
from row_segmenter import TransactionRowSegmenter

row_segmenter = TransactionRowSegmenter(filtered_words, bank_detector)

column_delimitation = row_segmenter.delimit_column_positions()
grouped_rows = row_segmenter.group_rows()

print(row_segmenter.row_threshold)
print(column_delimitation)
grouped_rows

7
{'column': ['FECHA', '(DE) (\\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (A) (\\d{2}) (ENE|FEB|MAR|ABR|MAY|JUN|JUL|AGO|SEP|OCT|NOV|DIC) (\\(\\d{2}) (DÍAS\\))', 'MONTO EN PESOS MEXICANOS'], 'x0': [58.0, None, 394.63000000000034], 'x1': [90.46, None, 544.4500000000004]}


,row_group,text,words,top,bottom,page
0,0,FECHA DEL 01 AL 30 SEP 2024 SEP 2024 (30 DÍAS)...,"[(FECHA, 58.0, 90.46), (DEL, 137.84, 155.81), ...",110.840,120.840,2
1,1,Depósito en Cajita: Mi Primera Cajita 29 SEP 2...,"[(Depósito, 135.84, 177.73), (en, 180.38, 192....",131.930,142.930,2
2,2,29 SEP 2024 SEP 2024 +$700.00 MARIO ALEXIS CAN...,"[(29 SEP 2024, 56.0, 114.84), (SEP, 70.39, 88....",158.020,169.520,2
3,3,"Depósito SPEI, Hora: 13:55:15, Recibido de BBV...","[(Depósito, 135.84, 173.54100000000003), (SPEI...",179.109,188.109,2
4,4,MARIO ALEXIS CANUDAS ZERTUCHE (Dato no verific...,"[(MARIO, 135.84, 165.054), (ALEXIS, 167.439, 1...",192.609,201.609,2
...,...,...,...,...,...,...
146,146,"2024090540014BMOV0000492839730, Clave de refer...","[(2024090540014BMOV0000492839730,, 135.84, 309...",520.129,529.129,5
147,147,1635110,"[(1635110, 135.84, 169.032)]",533.629,542.629,5
148,148,Pago a tu tarjeta de crédito Nu 03 SEP 2024 SE...,"[(Pago, 135.84, 159.35999999999999), (a, 162.0...",556.910,567.910,5
149,149,Retiro de Cajita: Mi Primera Cajita 03 SEP 202...,"[(Retiro, 135.84, 163.7), (de, 166.35, 178.3),...",583.000,594.000,5


In [6]:
from table_reconstructor import TransactionTableReconstructor

table_reconstructor = TransactionTableReconstructor(grouped_rows, column_delimitation, bank_detector)
words_classified = table_reconstructor.classify_columns()
words_classified

,Date,Description,Amount
0,30 SEP 2024,FECHA DEL 01 AL SEP 2024 (30 DÍAS) MONTO EN PE...,None
1,29 SEP 2024,Depósito en Cajita: Mi Primera Cajita SEP 2024,-$735.50
2,29 SEP 2024,SEP 2024 MARIO ALEXIS CANUDAS ZERTUCHE consulta,+$700.00
3,None,"Depósito SPEI, Hora: 13:55:15, Recibido de BBV...",None
4,None,MARIO ALEXIS CANUDAS ZERTUCHE (Dato no verific...,None
...,...,...,...
146,None,"2024090540014BMOV0000492839730, Clave de refer...",None
147,None,1635110,None
148,03 SEP 2024,Pago a tu tarjeta de crédito Nu SEP 2024,"-$2,377.49"
149,03 SEP 2024,Retiro de Cajita: Mi Primera Cajita SEP 2024,"+$2,378.00"


In [7]:
structured_table = table_reconstructor.get_structured_table()
structured_table

,Date,Description,MONTO EN PESOS MEXICANOS
0,30 SEP 2024,FECHA DEL 01 AL SEP 2024 (30 DÍAS) MONTO EN PE...,None
1,29 SEP 2024,Depósito en Cajita: Mi Primera Cajita SEP 2024,-$735.50
2,29 SEP 2024,SEP 2024 MARIO ALEXIS CANUDAS ZERTUCHE consulta,+$700.00
3,None,"Depósito SPEI, Hora: 13:55:15, Recibido de BBV...",None
4,None,MARIO ALEXIS CANUDAS ZERTUCHE (Dato no verific...,None
...,...,...,...
146,None,"2024090540014BMOV0000492839730, Clave de refer...",None
147,None,1635110,None
148,03 SEP 2024,Pago a tu tarjeta de crédito Nu SEP 2024,"-$2,377.49"
149,03 SEP 2024,Retiro de Cajita: Mi Primera Cajita SEP 2024,"+$2,378.00"


In [8]:
reconstructed_table = table_reconstructor.reconstruct_table()
reconstructed_table

,Date,Description,MONTO EN PESOS MEXICANOS
0,29 SEP 2024,Depósito en Cajita: Mi Primera Cajita SEP 2024,-$735.50
1,29 SEP 2024,SEP 2024 MARIO ALEXIS CANUDAS ZERTUCHE consult...,+$700.00
2,29 SEP 2024,CHICHI SUAREZ MERIDA Compra SEP 2024,-$64.50
3,29 SEP 2024,Retiro de Cajita: Mi Primera Cajita SEP 2024,+$100.00
4,27 SEP 2024,Depósito en Cajita: Mi Primera Cajita SEP 2024,"-$1,090.00"
5,27 SEP 2024,SEP 2024 MARIA FERNANDA CANUDAS ZERTUCHE Medic...,+$200.00
6,27 SEP 2024,FARMACIAS SIMILARES Compra SEP 2024,-$200.00
7,27 SEP 2024,Retiro de Cajita: Mi Primera Cajita SEP 2024,"+$1,000.00"
8,26 SEP 2024,PARRILLA GRAN PLAZA Compra SEP 2024,-$110.00
9,26 SEP 2024,Retiro de Cajita: Mi Primera Cajita SEP 2024,+$200.00


In [9]:
from table_normalizer import TransactionTableNormalizer

normalizer = TransactionTableNormalizer(reconstructed_table, bank_detector)

df_normalized = normalizer.normalize_table()
df_normalized

,Date,Description,Amount,Type
48,2024-09-03,Saldo Inicial,3559.80,Saldo Inicial
47,2024-09-03,Retiro de Cajita: Mi Primera Cajita SEP 2024 ...,2378.00,Abono
46,2024-09-03,Pago a tu tarjeta de crédito Nu SEP 2024,-2377.49,Cargo
45,2024-09-05,MARIO ANTONIO CANUDAS AREVALO TRANSFERENCIA A ...,1200.00,Abono
44,2024-09-05,Depósito en Cajita: Mi Primera Cajita SEP 2024,-1200.51,Cargo
43,2024-09-06,Retiro de Cajita: Mi Primera Cajita SEP 2024,270.00,Abono
42,2024-09-06,SEP 2024 jafet coba lira Transferencia Transf...,-150.00,Cargo
41,2024-09-06,SEP 2024 angel rosado Transferencia Transfere...,-120.00,Cargo
40,2024-09-08,SEP 2024 MARIO ALEXIS CANUDAS ZERTUCHE prueba ...,17.00,Abono
39,2024-09-08,SEP 2024 MARIO ALEXIS CANUDAS ZERTUCHE sully 2...,1000.50,Abono


In [10]:
normalizer.period_idx

In [11]:
from data_exporter import CsvExporter
from config import OUTPUTS_FOLDER

exporter = CsvExporter(df_normalized)
file_path = os.path.join(OUTPUTS_FOLDER, bank, f'{bank}_{statement_type}.csv')

exporter.export_to_csv(file_path)

Data exported to c:\Users\mario\Proyectos\Aether\Aether_v1\documents\outputs\nu\nu_debit.csv successfully.
